# Approach B — TRAINED continuous RT head

The real thing: a regression head (predicts (mu, log-sigma) of log-RT) **trained
jointly** with the choice loss (head + LoRA updated together), so the representation
is shaped to encode RT. Loss = choice CE + lam * log-normal NLL.

Individual signal at eval = per-trial **standardised residual** (logRT - mu)/sigma
(*faster / slower than the model expects*), summarised per person -> cross-task
identification.

Uses `output_nl_rtval` (choice-only text + continuous-RT sidecar) -- same data as the
probe notebook. No Liger here (the head hooks the lm_head input), so batch is small.

In [ ]:
# setup (+ unzip the RT-value data if needed)
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/sro-minitaur-transfer'):
    !cd /content/sro-minitaur-transfer && git pull
else:
    !git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git /content/sro-minitaur-transfer
!pip -q install peft bitsandbytes scikit-learn
RTV = '/content/drive/MyDrive/sro_minitaur/output_nl_rtval'
if not os.path.isdir(RTV + '/complete'):
    !cd /content/drive/MyDrive/sro_minitaur && unzip -oq output_nl_rtval.zip
assert os.path.isdir(RTV + '/complete'), 'upload output_nl_rtval.zip to MyDrive/sro_minitaur/ first'
print('RT-value tasks:', len(os.listdir(RTV + '/complete')))

In [ ]:
# 0) SMOKE TEST: confirm it trains without error (~10 min, 30 steps).
#    Watch: loss + rt_nll should both drop. If it runs clean, do the full run below.
!cd /content/sro-minitaur-transfer && python scripts/train_rt_head.py \
    --subset starting_subset \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rtval \
    --out /content/drive/MyDrive/sro_minitaur/mpop_rt_head_smoke \
    --batch-size 4 --grad-accum 4 --max-steps 30

In [ ]:
# 1) FULL TRAIN: head + LoRA, all 33 tasks, 1 epoch (~6-10 h on A100-40G).
#    Checkpoints adapter+head to Drive every 200 steps. A100-40G: batch 4 (no liger).
#    NOTE: the manual loop does not auto-resume -> for an uninterrupted run consider AutoDL.
!cd /content/sro-minitaur-transfer && python scripts/train_rt_head.py \
    --subset all \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rtval \
    --out /content/drive/MyDrive/sro_minitaur/mpop_rt_head \
    --batch-size 4 --grad-accum 4 --lam 1.0

In [ ]:
# 2) EVAL: residual transfer using the TRAINED head (--trained).
!cd /content/sro-minitaur-transfer && python scripts/rt_head_transfer.py \
    --trained /content/drive/MyDrive/sro_minitaur/mpop_rt_head \
    --nl-dir /content/drive/MyDrive/sro_minitaur/output_nl_rtval \
    --subset all --max-seq-len 4096

## How to read

Eval prints `within-domain mean top1` / `across` (chance 0.10). Compare:

| | within |
|---|---|
| choice-only surprise | ~0.15 |
| A — bin RT (CE) | 0.143–0.165 |
| B — trained RT head | **?** |

- **B > A** → training a continuous RT head extracts more transferable individual
  signal than 10-bin tokens (your instinct was right) — the headline RT result.
- **B ≈ A** → continuous didn't help beyond bins; RT's transferable signal is just thin.

During training, **`rt_nll` dropping** = the head is learning RT; if it doesn't move,
lower `--lr` or raise `--lam`.